In [1]:
import numpy as np
import pandas as pd
import requests

In [143]:
def fetch_moex_history(ticker, t_type, offset=0,  start_date="2024-01-03", limit=100, cols="TRADEDATE,CLOSE,NUMTRADES"):
    if t_type=="stock":
        url = f"https://iss.moex.com/iss/history/engines/stock/markets/shares/boards/TQBR/securities/{ticker}.json"
    elif t_type=="currency":
        url = f"https://iss.moex.com/iss/history/engines/currency/markets/selt/boards/CETS/securities/{ticker}.json"
    else:
        raise Exception("something is wrong..")
    params = {
        "from": start_date,
        "start": 1+offset,
        "iss.meta": "off",
        "history.columns": cols,
        "limit": limit,
    }
    res = requests.get(url, params=params).json()
    df = pd.DataFrame(res["history"]["data"], columns=res["history"]["columns"])
    df["TRADEDATE"] = pd.to_datetime(df["TRADEDATE"])
    return df

In [46]:
currency = "CNYRUB_TOM"
stocks = ["SBER", "LKOH", "GAZP", "PLZL", "NVTK", "YNDX"]

In [52]:
df = fetch_moex_history("SBER","stock","2024-01-03")

In [50]:
df['ch']=df['CLOSE'].pct_change()

In [ ]:
# volatility = prices["change"].std()  # Волатильность
# avg_turnover = prices["VOLUME"].mean()  # Ликвидность

In [ ]:
def find_currencies(query) -> pd.DataFrame:
    url = "https://iss.moex.com/iss/securities.json"
    params = {
        "q" : query,
        "iss.meta": "off"
    }
    res = requests.get(url, params=params).json()
    columns = res["securities"]["columns"]
    data = res["securities"]["data"]

    df = pd.DataFrame(data, columns=columns)
    df = df[ (df['group']=='currency_selt') & (df["primary_boardid"]=='CETS') 
            & (df['is_traded'])]
    return df[['secid','shortname','name']]




In [125]:
find_currencies("CNY")

,secid,shortname,name
0,CNYRUB_TOM,CNYRUB_TOM,CNY/RUB_TOM - CNY/РУБ
1,CNY000000TOD,CNYRUB_TOD,CNY/RUB_TOD - CNY/РУБ
2,CNYRUBTODTOM,CNY_TODTOM,CNY_TODTOM - СВОП CNY/РУБ
3,CNYRUB_TMS,CNYRUB_TMS,CNY/RUB_TMS - CNY/РУБ
6,CNYRUB_SPT,CNYRUB_SPT,CNY/RUB_SPT - CNY/РУБ
8,CNYRUB_TOM1D,CNY_TOMSPT,CNY_TOMSPT СВОП CNY/РУБ
11,USDCNYTODTOM,USDCNYTDTM,USD/CNY_TODTOM - СВОП USD/CNY
12,USDCNY_SPT,USDCNY_SPT,USD/CNY_SPT - USD/CNY
13,USDCNY_TOD,USDCNY_TOD,USD/CNY_TOD - USD/CNY
14,USDCNY_TOM,USDCNY_TOM,USD/CNY_TOM - USD/CNY


In [113]:
def find_stocks(query: str) -> pd.DataFrame:
    url = "https://iss.moex.com/iss/securities.json"
    params = {
        "q": query,
        "iss.meta": "off"
    }
    res = requests.get(url, params=params).json()
    df = pd.DataFrame(res["securities"]["data"], columns=res["securities"]["columns"])
    return df[(df["group"]=="stock_shares") & df["is_traded"]  
          & (df["primary_boardid"]=="TQBR") & 
          (df["marketprice_boardid"]=="TQBR")][["secid","shortname","name","emitent_title"]]

In [114]:
find_stocks("газ").head()

,secid,shortname,name,emitent_title
12,GAZP,ГАЗПРОМ ао,"""Газпром"" (ПАО) ао","Публичное акционерное общество ""Газпром"""
13,SIBN,Газпрнефть,Газпром нефть ПАО ао,"Публичное акционерное общество ""Газпром нефть"""
14,RTGZ,ГР Ростов,Газпром газорасп Р-н-Д ПАО ао,"Публичное акционерное общество ""Газпром газора..."
15,GAZA,ГАЗ ао,ГАЗ ПАО ао,"Публичное акционерное общество ""ГАЗ"""
16,GAZAP,ГАЗ ап,ГАЗ ПАО ап,"Публичное акционерное общество ""ГАЗ"""


In [131]:
find_stocks("OZON")

,secid,shortname,name,emitent_title
6,OZON,Озон,МКПАО Озон,Международная компания Публичное акционерное о...
7,OZPH,iОзонФарм,Озон Фармацевтика,"Публичное акционерное общество ""Озон Фармацевт..."


In [145]:
fetch_moex_history("OZON","stock",0,"2025-01-03")

,TRADEDATE,CLOSE,NUMTRADES
0,2025-01-06,3169.0,9479
1,2025-01-08,3152.5,8418
2,2025-01-09,3069.5,17523
3,2025-01-10,3192.0,20818
4,2025-01-13,3160.0,30253
...,...,...,...
95,2025-05-22,3516.0,5309
96,2025-05-23,3529.0,3582
97,2025-05-26,3443.0,6217
98,2025-05-27,3553.5,9113
